# Transfer Learning with VGG16

Two-phase transfer learning pipeline — Phase 1 (feature extraction) followed by Phase 2 (fine-tuning) — built on top of a pretrained VGG16 model.

## Overview

- **Base model:** VGG16, pretrained on ImageNet
- **Phase 1:** Freeze the entire base model, train only a new classification head
- **Phase 2:** Unfreeze the last few convolutional layers of the base model and fine-tune with a much smaller learning rate

## Requirements

In [ ]:
!pip install tensorflow -q

## Phase 1 — Feature Extraction

### Step 1: Load the pretrained base model

In [ ]:
import tensorflow as tf
from tensorflow import keras

base_model = keras.applications.VGG16(
    input_shape=(224, 224, 3),
    include_top=False,   # remove VGG16's original ImageNet classifier
    weights='imagenet'
)

### Step 2: Freeze the base

In [ ]:
base_model.trainable = False   # no weight updates in the base model during Phase 1

### Step 3: Attach a new head

In [ ]:
inputs = keras.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)          # extract features from the frozen base
x = keras.layers.GlobalAveragePooling2D()(x)     # feature maps -> flat vector
outputs = keras.layers.Dense(num_classes, activation='softmax')(x)  # new trainable head

model = keras.Model(inputs, outputs)

### Step 4: Compile

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

### Step 5: Train

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10
)

## Phase 2 — Fine-Tuning

### Step 1: Unfreeze the base model

In [ ]:
base_model.trainable = True

### Step 2: Re-freeze all but the last few layers

In [ ]:
# VGG16 has 19 layers total (feature-extraction layers)
fine_tune_at = len(base_model.layers) - 4   # unfreeze only the last 4 layers

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

### Step 3: Recompile with a much smaller learning rate

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # 100x smaller than Phase 1
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

### Step 4: Continue training

In [ ]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    initial_epoch=history.epoch[-1]   # continue epoch count from Phase 1
)

## Callbacks (optional, recommended)

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True
)

Pass these into either `fit()` call:

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

## Data Augmentation (optional, recommended for small datasets)

In [ ]:
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
])

Insert right after the input layer:

In [ ]:
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)        # active only during training
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
outputs = keras.layers.Dense(num_classes, activation='softmax')(x)
model = keras.Model(inputs, outputs)

## Full Pipeline Summary

1. Load pretrained VGG16 with `include_top=False`
2. Freeze the entire base model
3. Attach a new head (GlobalAveragePooling2D + Dense)
4. Compile and train — **Phase 1**
5. Unfreeze the last few layers of the base model
6. Recompile with a much smaller learning rate
7. Continue training — **Phase 2**
8. Use callbacks (EarlyStopping, ModelCheckpoint) to monitor and save the best model